In [32]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [33]:
df1 = pd.read_csv("./ssm_all_features.csv")
df2 = pd.read_csv("./features_before_selection.csv")

In [34]:
joined_df= pd.merge(df1, df2, on=["conv_id", "turn_id"], how="inner")

In [35]:
assert len(joined_df) == len(df1) == len(df2)
assert (joined_df["y"]==joined_df["label"]).all()

In [36]:
vector_list= list(str(col) for col in range(768+768))

In [37]:
vector_df= joined_df[vector_list]
feature_df= joined_df.drop(columns=vector_list+["label","y"])
y= joined_df["y"]

In [7]:
feature_df.columns

Index(['state_drift', 'state_input_distance', 'long_term_state_drift',
       'state_similarity', 'state_input_similarity',
       'long_term_state_similarity', 'conv_id', 'turn_id', 'toxicity_score',
       'threat_score', 'topic_drift_score', 'drift_acceleration',
       'interaction_risk', 'pattern_risk', 'progressive_risk',
       'prev_progressive', 'toxicity_score_ema3',
       'toxicity_score_rolling3_mean', 'toxicity_score_rolling3_max',
       'threat_score_ema3', 'threat_score_rolling3_mean',
       'threat_score_rolling3_max', 'interaction_risk_ema3',
       'interaction_risk_rolling3_mean', 'interaction_risk_rolling3_max',
       'pattern_risk_ema3', 'pattern_risk_rolling3_mean',
       'pattern_risk_rolling3_max', 'toxicity_diff', 'threat_diff',
       'toxicity_accel', 'risk_slope_3', 'max_toxicity_so_far',
       'max_threat_so_far', 'mean_risk_so_far', 'early_high_risk',
       'late_risk_increase', 'risk_growth_ratio'],
      dtype='object')

In [38]:
train_vectors, test_vectors, train_features, test_features, train_labels, test_labels = train_test_split(
    vector_df, feature_df, y, test_size=0.1, random_state=42
)

train_vectors,val_vectors, train_features, val_features, train_labels, val_labels = train_test_split(
    train_vectors, train_features, train_labels, test_size=0.1, random_state=42
)   

In [39]:
train_tensor = torch.tensor(
    [
        [v.item() if hasattr(v, "item") else float(v) for v in row]
        for row in train_vectors.values
    ],
    dtype=torch.float32
)

ValueError: could not convert string to float: 'tensor(0.0226)'

In [20]:
train_vectors = train_vectors.apply(pd.to_numeric, errors='coerce')
val_vectors   = val_vectors.apply(pd.to_numeric, errors='coerce')
test_vectors  = test_vectors.apply(pd.to_numeric, errors='coerce')

In [24]:
train_vectors.isnull().sum().sum(), val_vectors.isnull().sum().sum(), test_vectors.isnull().sum().sum()

(np.int64(4048896), np.int64(450048), np.int64(500736))

In [25]:
train_vectors.head()

,0,1,2,3,4,5,6,7,8,9,...,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535
2395,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
train_tensor = torch.tensor(train_vectors.values, dtype=torch.float32)
val_tensor   = torch.tensor(val_vectors.values, dtype=torch.float32)
test_tensor  = torch.tensor(test_vectors.values, dtype=torch.float32)

In [22]:
pca = PCA(n_components=2)   # reduce to 2D for plotting
reduced = pca.fit_transform(train_vectors)

plt.figure(figsize=(8,6))
plt.scatter(reduced[:, 0], reduced[:, 1], s=10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection of Train Vectors")
plt.show()

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values